# Исторический код автора

Снимок исходной исследовательской логики. Outputs, attachments и метаданные ядра очищены.
Это не основной маршрут запуска; исходная закрытая Excel-выгрузка не включена в проект.
Исторические обозначения Fine–Gray могут быть методологически неточными: см. docs/methods.md.
Для текущего проекта используйте notebooks/01_prepare_data.ipynb и 02_survival_analysis.ipynb.


In [ ]:
from pathlib import Path
import sys, os
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "configs/project.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Откройте ноутбук внутри cardio-oncology-survival-analysis")
sys.path.insert(0, str(ROOT / "src"))
from cardio_oncology.paths import project_paths
P = project_paths(ROOT)
P.run.mkdir(parents=True, exist_ok=True)
P.processed.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = P.run
os.environ.setdefault("MPLCONFIGDIR", str(OUTPUT_DIR / ".matplotlib"))


# Подготовка данных: Кардиоваскулярные осложнения при РТК
**Датасет:** Анализ_РТК_с_ССО.xlsx | Новосибирск, 6 мес | n = 257

Этапы:
1. Технический аудит
2. Удаление пустых столбцов и дублей
3. Очистка дат → возраст на начало лечения
4. Стандартизация пола
5. Временна́я переменная и статус цензурирования
6. Составные исходы (composite endpoints)
7. QTc (с → мс) и ΔQTc
8. Очистка числовых предикторов (запятые, нули, выбросы)
9. Кодирование категориальных переменных
10. Флаг бевацизумаба
11. Итоговая выборка для анализа выживаемости
12. Описательная статистика
13. Сохранение

## 0. Импорт библиотек

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from datetime import datetime

warnings.filterwarnings('ignore')
sns.set(style='whitegrid')
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.3f}'.format)

FILE_PATH = 'Анализ_РТК_с_ССО__1_.xlsx'
SHEET     = 'Основная таблица'
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## 1. Загрузка данных

In [ ]:
df_raw = pd.read_excel(FILE_PATH, sheet_name=SHEET)
print(f'Размер исходного датасета: {df_raw.shape[0]} пациентов × {df_raw.shape[1]} столбцов')
df_raw.head(3)

## 2. Технический аудит — обзор качества данных

In [ ]:
# ── 2.1 Пропуски по каждому столбцу ──────────────────────────────────────────
miss = df_raw.isnull().sum()
miss_pct = (miss / len(df_raw) * 100).round(1)
audit = pd.DataFrame({'missing_n': miss, 'missing_%': miss_pct, 'dtype': df_raw.dtypes})
audit = audit[audit['missing_n'] > 0].sort_values('missing_%', ascending=False)
print(f'Столбцов с пропусками: {len(audit)}')
display(audit)

In [ ]:
# ── 2.2 Полностью пустые столбцы (Unnamed) ───────────────────────────────────
empty_cols = [c for c in df_raw.columns if df_raw[c].isna().all()]
print(f'Полностью пустых столбцов: {len(empty_cols)}')
print(empty_cols[:10], '...')

In [ ]:
# ── 2.3 Варианты написания пола ──────────────────────────────────────────────
print('Уникальные значения пола:')
print(df_raw['пол'].value_counts(dropna=False))

In [ ]:
# ── 2.4 Распределение целевой переменной (кардиотоксичность) ─────────────────
cardiotox_col = 'Кардиотокс(1-Да,0-Нет)'
vc = df_raw[cardiotox_col].value_counts()
print('Кардиотоксичность:')
print(f'  Есть (1): {vc.get(1, 0)}  ({vc.get(1, 0)/len(df_raw)*100:.1f}%)')
print(f'  Нет  (0): {vc.get(0, 0)}  ({vc.get(0, 0)/len(df_raw)*100:.1f}%)')
print()
print('→ Дисбаланс классов минимален (~48/52%), undersampling не требуется.')
print('  Основные методы: Kaplan–Meier, Cox regression, Fine–Gray (конкурирующие риски).')

## 3. Удаление пустых столбцов и дублей

In [ ]:
df = df_raw.copy()

# ── 3.1 Удаляем полностью пустые столбцы (все 33 Unnamed) ────────────────────
df = df.drop(columns=empty_cols)
print(f'После удаления пустых столбцов: {df.shape[1]} столбцов')

# ── 3.2 Дублирующийся столбец Stage.1 (идентичен Stage) ─────────────────────
# Проверка идентичности перед удалением:
assert (df['Stage'] == df['Stage.1']).all(), 'Stage и Stage.1 различаются!'
df = df.drop(columns=['Stage.1'])
print('Stage.1 удалён (дубль Stage)')

# ── 3.3 Дублирующийся столбец Таргет.1 (идентичен Таргет) ───────────────────
col_target_orig = 'Таргет(1-Бев,2-Рег,3-Цет/Пан,4-Комби)'
col_target_dup  = 'Таргет(1-Бев,2-Рег,3-Цет/Пан,4-Комби).1'
if col_target_dup in df.columns:
    assert (df[col_target_orig] == df[col_target_dup]).all(), 'Таргет-столбцы различаются!'
    df = df.drop(columns=[col_target_dup])
    print('Таргет.1 удалён (дубль)')

# ── 3.4 Нормализация пробелов в именах столбцов ──────────────────────────────
df.columns = df.columns.str.strip().str.replace(r'\s+', ' ', regex=True)

print(f'\nИтого столбцов после очистки: {df.shape[1]}')

## 4. Даты и возраст

In [ ]:
def parse_date_mixed(val):
    """
    Парсит дату из смешанного поля:
    - уже datetime → возвращает как есть
    - строка с датой ('27.09.2024', '10.01.24', 'ДД.ММ.ГГ', 'ДД.ММ.ГГГГ') → парсит
    - строка с диагнозом после даты ('18.02.2021 C18...') → берёт первый токен
    - всё остальное → NaT
    """
    if pd.isnull(val):
        return pd.NaT
    if isinstance(val, datetime):
        return val
    s = str(val).strip()
    # берём только первый токен (до пробела или ICD-кода)
    token = s.split()[0] if s else ''
    # убираем крайние точки/запятые
    token = token.strip('.,;')
    for fmt in ('%d.%m.%Y', '%d.%m.%y', '%Y-%m-%d', '%d/%m/%Y'):
        try:
            return datetime.strptime(token, fmt)
        except ValueError:
            continue
    return pd.NaT

In [ ]:
# ── 4.1 Дата рождения ────────────────────────────────────────────────────────
# pandas уже разобрал большинство строк как datetime;
# оставшиеся строки (как '19.02.1958') — обрабатываем через parse_date_mixed
df['dob'] = df['Дата рождения'].apply(parse_date_mixed)
print(f'Дата рождения: {df["dob"].notna().sum()} / {len(df)} распознано')
print(f'  Диапазон: {df["dob"].min().year} – {df["dob"].max().year}')

# ── 4.2 Дата выявления заболевания ───────────────────────────────────────────
df['diag_date'] = df['Дата выявления заболевания'].apply(parse_date_mixed)
print(f'\nДата диагноза: {df["diag_date"].notna().sum()} / {len(df)} распознано')
n_fail = df['diag_date'].isna().sum()
if n_fail > 0:
    print(f'  Не распознано ({n_fail} строк):')
    print(df.loc[df['diag_date'].isna(), 'Дата выявления заболевания'].values)

# ── 4.3 Дата смерти ──────────────────────────────────────────────────────────
df['death_date'] = df['дата смерти'].apply(parse_date_mixed)
print(f'\nДата смерти: {df["death_date"].notna().sum()} умерших')

In [ ]:
# ── 4.4 Возраст на момент выявления заболевания ──────────────────────────────
df['age'] = ((df['diag_date'] - df['dob']).dt.days / 365.25).round(1)

# Санитарная проверка: возраст должен быть в диапазоне 18–95
outlier_age = df[(df['age'] < 18) | (df['age'] > 100)]
if len(outlier_age) > 0:
    print('Подозрительные значения возраста:')
    display(outlier_age[['Номер пациента', 'Дата рождения', 'Дата выявления заболевания', 'age']])

print(df['age'].describe().round(1))

## 5. Стандартизация пола

In [ ]:
# М/м → 1 (мужчина), Ж/ж → 0 (женщина)
sex_map = {'м': 1, 'М': 1, 'ж': 0, 'Ж': 0}
df['sex'] = df['пол'].map(sex_map)    # NaN остаётся NaN для пропущенных

print('Пол после стандартизации:')
print(df['sex'].value_counts(dropna=False))
print('  (1 = мужчина, 0 = женщина)')

## 6. Временна́я переменная и статус цензурирования (survival analysis)

In [ ]:
# ── Концепция ─────────────────────────────────────────────────────────────────
# time  = «Дней до кардиотоксичности» — для всех пациентов:
#           у пациентов с событием → дни от начала лечения до первого ССО
#           у цензурированных       → дни наблюдения без события
# event = Кардиотокс(1=Да, 0=Нет) — индикатор события
#
# Конкурирующий риск: Общая смертность (смерть от рака до ССО)
#   competing_event = 1 если умер без ССО, 2 если ССО, 0 если цензурирован
# ──────────────────────────────────────────────────────────────────────────────

TIME_COL  = 'Дней до кардиотоксичности 0- нет даных'
EVENT_COL = 'Кардиотокс(1-Да,0-Нет)'
DEATH_COL = 'Общая смертность смерть'
CV_DEATH_COL = 'Сердечно-сосудистая смерть'

df['time_days']  = pd.to_numeric(df[TIME_COL], errors='coerce')
df['event_cardiotox'] = pd.to_numeric(df[EVENT_COL], errors='coerce').astype('Int64')
df['event_death']     = pd.to_numeric(df[DEATH_COL], errors='coerce').astype('Int64')
df['event_cv_death']  = pd.to_numeric(df[CV_DEATH_COL], errors='coerce').astype('Int64')

print('Время до события (дни):')
print(df['time_days'].describe().round(0))

print('\nРаспределение событий:')
print(f'  Кардиотоксичность:      {df["event_cardiotox"].sum()} ({df["event_cardiotox"].mean()*100:.1f}%)')
print(f'  Общая смертность:       {df["event_death"].sum()} ({df["event_death"].mean()*100:.1f}%)')
print(f'  ССС-смертность:         {df["event_cv_death"].sum()} ({df["event_cv_death"].mean()*100:.1f}%)')

In [ ]:
# ── Конкурирующий статус для Fine–Gray ───────────────────────────────────────
# 0 = цензура (жив без ССО)
# 1 = ССО (интересующее событие)
# 2 = смерть без ССО (конкурирующее событие)
def compute_competing_status(row):
    if row['event_cardiotox'] == 1:
        return 1  # ССО
    elif row['event_death'] == 1:
        return 2  # смерть без ССО
    else:
        return 0  # цензура

df['competing_status'] = df.apply(compute_competing_status, axis=1)

print('Конкурирующий статус (для Fine–Gray):')
print(df['competing_status'].value_counts().rename({0: 'цензура', 1: 'ССО', 2: 'смерть без ССО'}))
print()
print('Примечание: competing_status используется в lifelines.AalenJohansenFitter')
print('и scikit-survival для анализа конкурирующих рисков.')

## 7. Составные исходы (composite endpoints)

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# ГРУППА 1: Острые коронарные события
SSO_ACS = ['ИмспST', 'ИмбпсST', 'Нестабильная стенокардия']

# ГРУППА 2: Нарушения ритма (клинически значимые)
SSO_RHYTHM_MAJOR = ['ФП', 'ЖТ', 'ФЖ', 'полная AV-блокада']

# ГРУППА 3: Нарушения ритма (незначимые / доброкачественные)
SSO_RHYTHM_MINOR = [
    'Предсердные экстрасистолии', 'замедление A-V проведения',
    'ЖЭС', 'Миграция водителя ритма', 'синусовая тахикардия', 'Брадикардия синусовая'
]

# ГРУППА 4: Тромбоэмболия и сердечная недостаточность
SSO_THROMBO_HF = [
    'Декомпенсация ранее существующей СН',
    'Системные артериальные эмболии',
    'Венозные тромбо эмболии'
]

# ГРУППА 5: Прочие
SSO_OTHER = ['Развитие de novo ССЗ']

# ── Все SSO-столбцы ───────────────────────────────────────────────────────────
ALL_SSO_COLS = SSO_ACS + SSO_RHYTHM_MAJOR + SSO_RHYTHM_MINOR + SSO_THROMBO_HF + SSO_OTHER

# Конвертируем в числовой тип
for c in ALL_SSO_COLS:
    df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0).astype(int)

# ── Составные флаги ───────────────────────────────────────────────────────────
df['sso_any']          = (df[ALL_SSO_COLS].sum(axis=1) > 0).astype(int)
df['sso_acs']          = (df[SSO_ACS].sum(axis=1) > 0).astype(int)
df['sso_rhythm_major'] = (df[SSO_RHYTHM_MAJOR].sum(axis=1) > 0).astype(int)
df['sso_rhythm_minor'] = (df[SSO_RHYTHM_MINOR].sum(axis=1) > 0).astype(int)
df['sso_thrombo_hf']   = (df[SSO_THROMBO_HF].sum(axis=1) > 0).astype(int)

# Сводная таблица
sso_summary = pd.DataFrame({
    'Группа': ['Любое ССО', 'ОКС (ИМспST / ИМбпсST / НС)', 'Аритмии (клинически значимые)',
               'Аритмии (незначимые)', 'Тромбоз/ХСН'],
    'n':   [df[f].sum() for f in ['sso_any','sso_acs','sso_rhythm_major','sso_rhythm_minor','sso_thrombo_hf']],
    '%':   [(df[f].mean()*100).round(1) for f in ['sso_any','sso_acs','sso_rhythm_major','sso_rhythm_minor','sso_thrombo_hf']]
})
display(sso_summary)

print()
print('Частота отдельных ССО-событий:')
sso_individual = df[ALL_SSO_COLS].sum().sort_values(ascending=False)
display(pd.DataFrame({'n': sso_individual, '%': (sso_individual/len(df)*100).round(1)}))

## 8. QTc: конвертация с→мс и вычисление ΔQTc

In [ ]:
def clean_qtc(series):
    """
    Очищает столбец QTc:
    - заменяет запятую на точку ('0,38' → '0.38')
    - парсит в float
    - если значение < 1 → умножает на 1000 (секунды → миллисекунды)
    - если значение уже в мс (например, 380) → оставляет
    Норма QTc: 300–500 мс. Всё вне [250, 700] считается ошибкой → NaN.
    """
    result = (series.astype(str)
              .str.strip()
              .str.replace(',', '.', regex=False)
              .pipe(pd.to_numeric, errors='coerce'))
    # значения < 1 → в секундах, умножаем на 1000
    result = result.where(result >= 1, result * 1000)
    # явные ошибки — вне клинически допустимого диапазона
    result = result.where((result >= 250) & (result <= 700))
    return result.round(0)

In [ ]:
df['qtc_start_ms'] = clean_qtc(df['QTc начало в секундах'])
df['qtc_last_ms']  = clean_qtc(df['QTc последнее в секундах'])

# ΔQTc = изменение за период наблюдения (положительное → удлинение, риск аритмии)
df['delta_qtc_ms'] = df['qtc_last_ms'] - df['qtc_start_ms']

# Флаги клинически значимых пороговых значений
# Норма: мужчины < 440 мс, женщины < 460 мс
qtc_thresh = df['sex'].map({1: 440, 0: 460}).fillna(450)   # 450 если пол неизвестен
df['qtc_start_prolonged'] = (df['qtc_start_ms'] >= qtc_thresh).astype('Int64')
df['qtc_last_prolonged']  = (df['qtc_last_ms']  >= qtc_thresh).astype('Int64')
df['delta_qtc_gt30']      = (df['delta_qtc_ms'] >= 30).astype('Int64')   # порог настороженности
df['delta_qtc_gt60']      = (df['delta_qtc_ms'] >= 60).astype('Int64')   # порог коррекции терапии

print('QTc начальный (мс):', df['qtc_start_ms'].describe().round(1).to_dict())
print('QTc последний (мс):', df['qtc_last_ms'].describe().round(1).to_dict())
print('ΔQTc (мс):          ', df['delta_qtc_ms'].describe().round(1).to_dict())
print()
print(f'Исходно удлинённый QTc: {df["qtc_start_prolonged"].sum()} пациентов')
print(f'ΔQTc > 30 мс:           {df["delta_qtc_gt30"].sum()} пациентов')
print(f'ΔQTc > 60 мс:           {df["delta_qtc_gt60"].sum()} пациентов')
print(f'QTc пропуски: {df["qtc_start_ms"].isna().sum()} / {len(df)}')

## 9. Очистка числовых предикторов

In [ ]:
def clean_numeric_ru(series, zero_as_nan=False, lower_bound=None, upper_bound=None):
    """
    Универсальная очистка числового столбца для русскоязычных данных:
    - Заменяет запятую на точку ('26,6' → 26.6)
    - Парсит в float, нечисловые → NaN
    - zero_as_nan=True: нулевые значения → NaN (физически невозможные нули)
    - lower_bound / upper_bound: значения вне диапазона → NaN
    """
    result = (series.astype(str)
              .str.strip()
              .str.replace(',', '.', regex=False)
              .pipe(pd.to_numeric, errors='coerce'))
    if zero_as_nan:
        result = result.replace(0, np.nan)
    if lower_bound is not None:
        result = result.where(result >= lower_bound)
    if upper_bound is not None:
        result = result.where(result <= upper_bound)
    return result


def winsorize_col(series, lower_q=0.025, upper_q=0.975):
    """
    Ограничение выбросов на уровне квантилей (winsorizing).
    Стандартный подход из ОКС-исследования — сохранён без изменений.
    """
    lo = series.quantile(lower_q)
    hi = series.quantile(upper_q)
    return series.clip(lower=lo, upper=hi)

In [ ]:
# ── 9.1 ИМТ ──────────────────────────────────────────────────────────────────
# Имеет запятые ('26,6') и одно нулевое значение (ошибка ввода)
df['bmi'] = clean_numeric_ru(df['ИМТ'], zero_as_nan=True, lower_bound=10, upper_bound=60)
print(f'ИМТ: {df["bmi"].notna().sum()} / {len(df)} | range: {df["bmi"].min():.1f}–{df["bmi"].max():.1f}')

# ── 9.2 ЧСС (ритм сердца) ────────────────────────────────────────────────────
df['hr'] = clean_numeric_ru(df['ЧСС'], lower_bound=20, upper_bound=250)
print(f'ЧСС: {df["hr"].notna().sum()} / {len(df)} | range: {df["hr"].min():.0f}–{df["hr"].max():.0f}')

# ── 9.3 СОЭ (начальная) ──────────────────────────────────────────────────────
df['esr_start'] = clean_numeric_ru(df['соэ начало'], lower_bound=1, upper_bound=120)
print(f'СОЭ нач.: {df["esr_start"].notna().sum()} / {len(df)}')

# ── 9.4 Гемоглобин (начальный) ───────────────────────────────────────────────
df['hgb_start'] = clean_numeric_ru(df['гемоглобин в начале заболевания'], lower_bound=50, upper_bound=200)
print(f'Гемоглобин нач.: {df["hgb_start"].notna().sum()} / {len(df)}')

# ── 9.5 Глюкоза (начальная) ──────────────────────────────────────────────────
df['glucose_start'] = clean_numeric_ru(df['глюоза на момент начала з-я'],
                                       zero_as_nan=True, lower_bound=2, upper_bound=35)
print(f'Глюкоза нач.: {df["glucose_start"].notna().sum()} / {len(df)}')

# ── 9.6 ФВ ЛЖ (%) ────────────────────────────────────────────────────────────
df['lvef'] = clean_numeric_ru(df['ФВ %'], lower_bound=10, upper_bound=85)
print(f'ФВ ЛЖ (%): {df["lvef"].notna().sum()} / {len(df)}')

# ── 9.7 Размер левого предсердия (мм) ────────────────────────────────────────
df['la_size'] = clean_numeric_ru(df['Размер левого передсердия'], lower_bound=20, upper_bound=65)
print(f'Размер ЛП (мм): {df["la_size"].notna().sum()} / {len(df)}')

# ── 9.8 СРБ (начальный) ──────────────────────────────────────────────────────
df['crp_start'] = clean_numeric_ru(df['СРБ на момент начала заболевания'],
                                   zero_as_nan=False, lower_bound=0)
print(f'СРБ нач.: {df["crp_start"].notna().sum()} / {len(df)}')

# ── 9.9 Лейкоциты (начальные) ─────────────────────────────────────────────────
df['wbc_start'] = clean_numeric_ru(df['лейоциты на начало'], lower_bound=1, upper_bound=50)
print(f'Лейкоциты нач.: {df["wbc_start"].notna().sum()} / {len(df)}')

# ── 9.10 Мочевая кислота (начальная) ─────────────────────────────────────────
df['uric_acid_start'] = clean_numeric_ru(df['мочевая кислота на момент началааболевания'],
                                         zero_as_nan=True, lower_bound=50, upper_bound=1200)
print(f'Мочевая кислота нач.: {df["uric_acid_start"].notna().sum()} / {len(df)}')

# ── 9.11 Холестерин (начальный) ──────────────────────────────────────────────
df['chol_start'] = clean_numeric_ru(df['холестерин нач'], zero_as_nan=True, lower_bound=1, upper_bound=15)
print(f'Холестерин нач.: {df["chol_start"].notna().sum()} / {len(df)}')

In [ ]:
# ── 9.12 Winsorizing числовых предикторов (квантили 2.5% / 97.5%) ─────────────
# Применяем к переменным с известными правосторонними выбросами
COLS_TO_WINSORIZE = [
    'hr', 'esr_start', 'crp_start', 'uric_acid_start',
    'wbc_start', 'hgb_start', 'glucose_start',
    'qtc_start_ms', 'qtc_last_ms', 'delta_qtc_ms'
]

for col in COLS_TO_WINSORIZE:
    if col in df.columns:
        before_range = f'{df[col].min():.1f}–{df[col].max():.1f}'
        df[col] = winsorize_col(df[col])
        after_range  = f'{df[col].min():.1f}–{df[col].max():.1f}'
        if before_range != after_range:
            print(f'{col:<25} до: {before_range:<20} после: {after_range}')

print('Winsorizing завершён.')

In [ ]:
# ── 9.13 Снижение ФВ < 50% (клинический порог) ────────────────────────────────
df['lvef_reduced'] = (df['lvef'] < 50).astype('Int64')   # сниженная ФВ
print(f'Сниженная ФВ < 50%: {df["lvef_reduced"].sum()} ({df["lvef_reduced"].mean()*100:.1f}%)')

# ── 9.14 Анемия (Hb < 120 г/л у женщин, < 130 г/л у мужчин) ─────────────────
hgb_thresh = df['sex'].map({1: 130, 0: 120}).fillna(125)
df['anemia'] = (df['hgb_start'] < hgb_thresh).astype('Int64')
print(f'Анемия на старте:    {df["anemia"].sum()} ({df["anemia"].mean()*100:.1f}%)')

## 10. Кодирование категориальных переменных

In [ ]:
# ── 10.1 Стадия РТК ──────────────────────────────────────────────────────────
# Stage: -1 (неизвестно), 0 (не определена) → NaN; 1/2/3/4 → оставляем
df['stage'] = pd.to_numeric(df['Stage'], errors='coerce')
df['stage'] = df['stage'].where(df['stage'] > 0)   # -1 и 0 → NaN

# Бинарный флаг: поздняя стадия (III–IV vs I–II)
df['stage_advanced'] = (df['stage'] >= 3).astype('Int64')

print('Стадия:')
print(df['stage'].value_counts(dropna=False))
print(f'\nПоздняя стадия (III–IV): {df["stage_advanced"].sum()}')

In [ ]:
# ── 10.2 Схема химиотерапии ───────────────────────────────────────────────────
# 0 = нет ХТ, 1=FOLFOX, 2=FOLFIRI, 3=XELOX, 4=DeGramont, 5=Другое
SCHEME_COL = 'Схема(1-FOLFOX,2-FOLFIRI,3-XELOX,4-DeGramont,5-Другое)'
df['cht_scheme'] = pd.to_numeric(df[SCHEME_COL], errors='coerce').astype('Int64')

SCHEME_LABELS = {
    0: 'Нет ХТ', 1: 'FOLFOX', 2: 'FOLFIRI',
    3: 'XELOX', 4: 'DeGramont', 5: 'Другое'
}
df['cht_scheme_label'] = df['cht_scheme'].map(SCHEME_LABELS)

# Флаг: схема с наибольшим кардиоваскулярным риском (платиносодержащая 5-ФУ)
# FOLFOX (1) и XELOX (3) содержат оксалиплатин + 5-ФУ/капецитабин
df['cht_high_cv_risk'] = df['cht_scheme'].isin([1, 3]).astype('Int64')

print('Схема ХТ:')
print(df['cht_scheme_label'].value_counts(dropna=False))

In [ ]:
# ── 10.3 Таргетная терапия и флаг бевацизумаба ────────────────────────────────
# 0=нет, 1=Бевацизумаб, 2=Регорафениб, 3=Цетуксимаб/Панитумумаб, 4=Комбинация
df['target_therapy'] = pd.to_numeric(df['Таргет(1-Бев,2-Рег,3-Цет/Пан,4-Комби)'],
                                     errors='coerce').astype('Int64')

# Бевацизумаб — антиVEGF, повышает риск тромбозов и гипертонии
df['bevacizumab'] = (df['target_therapy'] == 1).astype('Int64')

print('Таргетная терапия:')
TARGET_LABELS = {0:'Нет', 1:'Бевацизумаб', 2:'Регорафениб', 3:'Цет/Пан', 4:'Комби'}
print(df['target_therapy'].map(TARGET_LABELS).value_counts(dropna=False))
print(f'\nБевацизумаб: {df["bevacizumab"].sum()} ({df["bevacizumab"].mean()*100:.1f}%)')

In [ ]:
# ── 10.4 Платиносодержащие препараты ─────────────────────────────────────────
df['platinum'] = pd.to_numeric(
    df['Платиносодержащие препараты 0-нет, 1-да'], errors='coerce'
).astype('Int64')
print(f'Платиносодержащие: {df["platinum"].sum()} ({df["platinum"].mean()*100:.1f}%)')

# ── 10.5 Количество курсов XT ─────────────────────────────────────────────────
df['n_courses'] = pd.to_numeric(df['Курсов(кол-во)'], errors='coerce')
print(f'Курсов XT: медиана = {df["n_courses"].median()}, max = {df["n_courses"].max()}')

# ── 10.6 Гистология ───────────────────────────────────────────────────────────
# 1=G1 (высокодифф.), 2=G2 (умеренно), 3=G3 (низкодифф.), 99=НД
df['histology'] = pd.to_numeric(df['Гистология (1-G1, 2-G2, 3-G3, 99-НД)'], errors='coerce')
df['histology'] = df['histology'].where(df['histology'] != 99)   # 99 → NaN
print(f'\nГистология: {df["histology"].value_counts(dropna=False).to_dict()}')

# ── 10.7 ССЗ в анамнезе ───────────────────────────────────────────────────────
df['cvd_history'] = pd.to_numeric(df['ссз в анамнезе'], errors='coerce').astype('Int64')
print(f'\nССЗ в анамнезе: {df["cvd_history"].sum()} ({df["cvd_history"].mean()*100:.1f}%)')

# ── 10.8 Нарушение ритма (бинарный) ──────────────────────────────────────────
df['rhythm_disturbed'] = pd.to_numeric(df['Ритм(1-Аритм/Тах,0-Норм)'], errors='coerce').astype('Int64')
print(f'Нарушение ритма: {df["rhythm_disturbed"].sum()} ({df["rhythm_disturbed"].mean()*100:.1f}%)')

# ── 10.9 Метастазы в печень ───────────────────────────────────────────────────
df['mets_liver'] = pd.to_numeric(df['МТС_Печ (0-нет, 1-есть, 99-НД)'], errors='coerce')
df['mets_liver'] = df['mets_liver'].where(df['mets_liver'] != 99)   # 99 → NaN
print(f'Метастазы в печень: {df["mets_liver"].sum()} ({(df["mets_liver"]==1).mean()*100:.1f}%)')

## 11. Итоговый датасет для анализа

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Финальный набор признаков для анализа выживаемости (Cox, KM, Fine–Gray, RF)
# ──────────────────────────────────────────────────────────────────────────────

# ── Временны́е и целевые переменные ──────────────────────────────────────────
SURVIVAL_VARS = [
    'time_days',           # дни до события / цензуры
    'event_cardiotox',     # 1 = кардиотоксичность (основной исход)
    'event_death',         # 1 = смерть любой причины
    'event_cv_death',      # 1 = смерть от ССЗ
    'competing_status',    # 0/1/2 для Fine–Gray
]

# ── Составные ССО-флаги ───────────────────────────────────────────────────────
SSO_FLAGS = [
    'sso_any', 'sso_acs', 'sso_rhythm_major', 'sso_rhythm_minor', 'sso_thrombo_hf'
]

# ── Предикторы (ключевые — по заданию медиков) ────────────────────────────────
PREDICTORS = [
    # Демография
    'age', 'sex',
    # Клинический статус
    'hr', 'rhythm_disturbed', 'lvef', 'lvef_reduced', 'la_size',
    # Лабораторные — воспаление
    'esr_start', 'crp_start', 'wbc_start',
    # Лабораторные — метаболизм
    'bmi', 'hgb_start', 'anemia', 'glucose_start', 'chol_start',
    # QTc
    'qtc_start_ms', 'qtc_last_ms', 'delta_qtc_ms',
    'qtc_start_prolonged', 'delta_qtc_gt30', 'delta_qtc_gt60',
    # Онкологический статус
    'stage', 'stage_advanced', 'histology', 'mets_liver',
    # Лечение
    'cht_scheme', 'cht_scheme_label', 'cht_high_cv_risk',
    'platinum', 'n_courses',
    'target_therapy', 'bevacizumab',
    # Сопутствующие заболевания
    'cvd_history',
]

# ── Идентификация + исходные ССО-события ──────────────────────────────────────
ID_COLS = ['Номер пациента']
RAW_SSO_EVENTS = ALL_SSO_COLS

# ── Собираем финальный датасет ────────────────────────────────────────────────
all_cols = ID_COLS + SURVIVAL_VARS + SSO_FLAGS + PREDICTORS + RAW_SSO_EVENTS
# убираем дубли (если вдруг какой-то ССО-столбец попал дважды)
all_cols = list(dict.fromkeys(all_cols))

df_final = df[all_cols].copy()

print(f'Финальный датасет: {df_final.shape[0]} пациентов × {df_final.shape[1]} признаков')
df_final.head(3)

In [ ]:
# ── Санитарная проверка: пропуски в ключевых предикторах ──────────────────────
key_preds = ['age', 'sex', 'hr', 'lvef', 'bmi', 'hgb_start', 'glucose_start',
             'stage', 'n_courses', 'time_days', 'event_cardiotox']

miss_report = pd.DataFrame({
    'missing_n':   df_final[key_preds].isnull().sum(),
    'missing_%':   (df_final[key_preds].isnull().sum() / len(df_final) * 100).round(1)
})
print('Пропуски в ключевых предикторах:')
display(miss_report)

# QTc — отдельно (ожидаем ~43% пропусков)
print(f'\nQTc начало: {df_final["qtc_start_ms"].isna().sum()} пропусков ({df_final["qtc_start_ms"].isna().mean()*100:.1f}%)')
print('→ Анализ с QTc будет на подвыборке пациентов с ЭКГ-данными.')

## 12. Описательная статистика

In [ ]:
# ── 12.1 Таблица 1: непрерывные предикторы ────────────────────────────────────
continuous_preds = ['age', 'hr', 'bmi', 'lvef', 'la_size',
                    'hgb_start', 'glucose_start', 'esr_start', 'crp_start',
                    'n_courses', 'qtc_start_ms', 'delta_qtc_ms', 'time_days']

desc = df_final[continuous_preds].describe(percentiles=[0.25, 0.5, 0.75]).T
desc = desc[['count', 'mean', 'std', '25%', '50%', '75%', 'min', 'max']].round(2)
desc.columns = ['n', 'Mean', 'SD', 'Q1', 'Median', 'Q3', 'Min', 'Max']
print('Таблица 1 — Непрерывные предикторы (вся выборка):')
display(desc)

In [ ]:
# ── 12.2 Таблица 2: непрерывные предикторы — ССО vs без ССО ──────────────────
from scipy.stats import mannwhitneyu

rows = []
for col in continuous_preds:
    g0 = df_final.loc[df_final['event_cardiotox'] == 0, col].dropna()
    g1 = df_final.loc[df_final['event_cardiotox'] == 1, col].dropna()
    stat, p = mannwhitneyu(g0, g1, alternative='two-sided') if (len(g0) > 0 and len(g1) > 0) else (np.nan, np.nan)
    rows.append({
        'Признак': col,
        'Без ССО (Med [Q1-Q3])': f"{g0.median():.1f} [{g0.quantile(.25):.1f}–{g0.quantile(.75):.1f}]",
        'ССО (Med [Q1-Q3])':     f"{g1.median():.1f} [{g1.quantile(.25):.1f}–{g1.quantile(.75):.1f}]",
        'p (MW)': f'{p:.3f}' if not np.isnan(p) else 'NaN'
    })
display(pd.DataFrame(rows))

In [ ]:
# ── 12.3 Таблица 3: категориальные предикторы ─────────────────────────────────
from scipy.stats import chi2_contingency

cat_preds = ['sex', 'cvd_history', 'rhythm_disturbed', 'lvef_reduced', 'anemia',
             'stage_advanced', 'platinum', 'bevacizumab', 'cht_high_cv_risk',
             'qtc_start_prolonged', 'delta_qtc_gt30']

rows_cat = []
for col in cat_preds:
    ct = pd.crosstab(df_final['event_cardiotox'], df_final[col])
    if ct.shape == (2, 2):
        _, p, _, _ = chi2_contingency(ct)
        n_event  = int(df_final.loc[df_final['event_cardiotox']==1, col].sum())
        n_no_ev  = int(df_final.loc[df_final['event_cardiotox']==0, col].sum())
        tot_ev   = df_final['event_cardiotox'].sum()
        tot_no   = (df_final['event_cardiotox']==0).sum()
        rows_cat.append({
            'Признак (=1)': col,
            'Без ССО n (%)': f"{n_no_ev} ({n_no_ev/tot_no*100:.1f}%)",
            'ССО n (%)':     f"{n_event} ({n_event/tot_ev*100:.1f}%)",
            'p (χ²)': f'{p:.3f}'
        })
    else:
        rows_cat.append({'Признак (=1)': col, 'Без ССО n (%)': 'N/A', 'ССО n (%)': 'N/A', 'p (χ²)': 'N/A'})

display(pd.DataFrame(rows_cat))

In [ ]:
# ── 12.4 Распределение времени до события ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram времени
axes[0].hist(df_final.loc[df_final['event_cardiotox']==1, 'time_days'],
             bins=30, alpha=0.7, color='#D65F5F', label='ССО (event=1)')
axes[0].hist(df_final.loc[df_final['event_cardiotox']==0, 'time_days'],
             bins=30, alpha=0.5, color='#4878CF', label='Цензура (event=0)')
axes[0].set_xlabel('Дни до события / цензуры')
axes[0].set_ylabel('Число пациентов')
axes[0].set_title('Распределение времени наблюдения')
axes[0].legend()

# Boxplot по схемам ХТ
scheme_order = [s for s in ['FOLFOX','XELOX','FOLFIRI','DeGramont','Другое','Нет ХТ']
                if s in df_final['cht_scheme_label'].values]
df_plot = df_final.dropna(subset=['cht_scheme_label', 'time_days'])
df_plot.boxplot(column='time_days', by='cht_scheme_label', ax=axes[1],
                figsize=(7,5), grid=True)
axes[1].set_title('Время наблюдения по схемам ХТ')
axes[1].set_xlabel('Схема ХТ')
axes[1].set_ylabel('Дни')
plt.suptitle('')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'time_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('График сохранён: time_distribution.png')

In [ ]:
# ── 12.5 Распределение QTc ────────────────────────────────────────────────────
df_qtc = df_final.dropna(subset=['qtc_start_ms', 'qtc_last_ms'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# KDE QTc начало vs последнее
sns.kdeplot(df_qtc['qtc_start_ms'], ax=axes[0], label='QTc начало', color='#4878CF')
sns.kdeplot(df_qtc['qtc_last_ms'],  ax=axes[0], label='QTc последнее', color='#D65F5F')
axes[0].axvline(440, color='orange', linestyle='--', alpha=0.8, label='Порог 440 мс')
axes[0].set_xlabel('QTc (мс)')
axes[0].set_title('QTc начало vs последнее')
axes[0].legend()

# ΔQTc
color_event = df_qtc['event_cardiotox'].map({0: '#4878CF', 1: '#D65F5F'})
axes[1].scatter(df_qtc['qtc_start_ms'], df_qtc['delta_qtc_ms'],
                c=color_event, alpha=0.5, s=30)
axes[1].axhline(30, color='orange', linestyle='--', label='ΔQTc=30 мс')
axes[1].axhline(60, color='red',    linestyle='--', label='ΔQTc=60 мс')
axes[1].set_xlabel('QTc начало (мс)')
axes[1].set_ylabel('ΔQTc (мс)')
axes[1].set_title('ΔQTc vs QTc начало (синий=нет ССО, красный=ССО)')
axes[1].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'qtc_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'QTc на подвыборке n={len(df_qtc)} пациентов')

## 13. Сохранение

In [ ]:
# ── Основной датасет ─────────────────────────────────────────────────────────
df_final.to_csv(OUTPUT_DIR / 'РТК_ССО_подготовленный.csv', index=False, encoding='utf-8-sig')
print(f'Сохранён: РТК_ССО_подготовленный.csv  ({df_final.shape[0]} × {df_final.shape[1]})')

# ── Подвыборка с QTc (для QTc-специфических моделей) ─────────────────────────
df_qtc_subset = df_final.dropna(subset=['qtc_start_ms', 'qtc_last_ms'])
df_qtc_subset.to_csv(OUTPUT_DIR / 'РТК_ССО_QTc_подвыборка.csv', index=False, encoding='utf-8-sig')
print(f'Сохранён: РТК_ССО_QTc_подвыборка.csv  ({df_qtc_subset.shape[0]} × {df_qtc_subset.shape[1]})')

# ── Сводка по переменным ─────────────────────────────────────────────────────
var_dict = pd.DataFrame([
    # ЦЕЛЕВЫЕ
    {'Переменная': 'time_days',          'Тип': 'Время', 'Описание': 'Дни до события или цензуры'},
    {'Переменная': 'event_cardiotox',    'Тип': 'Событие', 'Описание': '1=кардиотоксичность (основной исход)'},
    {'Переменная': 'event_death',        'Тип': 'Событие', 'Описание': '1=смерть любой причины'},
    {'Переменная': 'competing_status',   'Тип': 'Статус', 'Описание': '0=цензура, 1=ССО, 2=смерть без ССО'},
    # ССО
    {'Переменная': 'sso_any',            'Тип': 'Исход', 'Описание': 'Любое ССО-событие'},
    {'Переменная': 'sso_acs',            'Тип': 'Исход', 'Описание': 'ОКС (ИМспST / ИМбпсST / НС)'},
    {'Переменная': 'sso_rhythm_major',   'Тип': 'Исход', 'Описание': 'Значимые аритмии (ФП, ЖТ, ФЖ, AV-блокада)'},
    # ПРЕДИКТОРЫ
    {'Переменная': 'age',                'Тип': 'Непрерывная', 'Описание': 'Возраст на момент диагноза (лет)'},
    {'Переменная': 'sex',                'Тип': 'Бинарная', 'Описание': '1=мужчина, 0=женщина'},
    {'Переменная': 'hr',                 'Тип': 'Непрерывная', 'Описание': 'ЧСС (уд/мин), winsorized'},
    {'Переменная': 'rhythm_disturbed',   'Тип': 'Бинарная', 'Описание': '1=аритмия/тахикардия исходно'},
    {'Переменная': 'lvef',               'Тип': 'Непрерывная', 'Описание': 'ФВ ЛЖ (%)'},
    {'Переменная': 'lvef_reduced',       'Тип': 'Бинарная', 'Описание': '1=ФВ<50%'},
    {'Переменная': 'la_size',            'Тип': 'Непрерывная', 'Описание': 'Размер левого предсердия (мм)'},
    {'Переменная': 'esr_start',          'Тип': 'Непрерывная', 'Описание': 'СОЭ начальное (мм/ч)'},
    {'Переменная': 'hgb_start',          'Тип': 'Непрерывная', 'Описание': 'Гемоглобин начальный (г/л)'},
    {'Переменная': 'anemia',             'Тип': 'Бинарная', 'Описание': '1=анемия (Hb<нормы по полу)'},
    {'Переменная': 'glucose_start',      'Тип': 'Непрерывная', 'Описание': 'Глюкоза начальная (ммоль/л)'},
    {'Переменная': 'bmi',                'Тип': 'Непрерывная', 'Описание': 'ИМТ (кг/м²)'},
    {'Переменная': 'qtc_start_ms',       'Тип': 'Непрерывная', 'Описание': 'QTc исходный (мс)'},
    {'Переменная': 'delta_qtc_ms',       'Тип': 'Непрерывная', 'Описание': 'ΔQTc (мс) = QTc_last − QTc_start'},
    {'Переменная': 'qtc_start_prolonged','Тип': 'Бинарная', 'Описание': '1=QTc исходно удлинён'},
    {'Переменная': 'delta_qtc_gt30',     'Тип': 'Бинарная', 'Описание': '1=ΔQTc≥30 мс'},
    {'Переменная': 'stage',              'Тип': 'Порядковая', 'Описание': 'Стадия РТК (1–4)'},
    {'Переменная': 'stage_advanced',     'Тип': 'Бинарная', 'Описание': '1=III–IV стадия'},
    {'Переменная': 'cht_scheme',         'Тип': 'Категориальная', 'Описание': '0=нет, 1=FOLFOX, 2=FOLFIRI, 3=XELOX...'},
    {'Переменная': 'cht_high_cv_risk',   'Тип': 'Бинарная', 'Описание': '1=FOLFOX/XELOX (высокий ССР)'},
    {'Переменная': 'platinum',           'Тип': 'Бинарная', 'Описание': '1=платиносодержащая схема'},
    {'Переменная': 'n_courses',          'Тип': 'Непрерывная', 'Описание': 'Количество курсов ХТ'},
    {'Переменная': 'bevacizumab',        'Тип': 'Бинарная', 'Описание': '1=получал бевацизумаб'},
    {'Переменная': 'cvd_history',        'Тип': 'Бинарная', 'Описание': '1=ССЗ в анамнезе'},
])
var_dict.to_csv(OUTPUT_DIR / 'РТК_ССО_словарь_переменных.csv', index=False, encoding='utf-8-sig')
print('Сохранён: РТК_ССО_словарь_переменных.csv')

print('\n=== Подготовка данных завершена ===' )
print(f'  Пациентов:             {len(df_final)}')
print(f'  Признаков в модели:    {len(PREDICTORS)}')
print(f'  Кардиотоксичность:     {df_final["event_cardiotox"].sum()} ({df_final["event_cardiotox"].mean()*100:.1f}%)')
print(f'  Медиана наблюдения:    {df_final["time_days"].median():.0f} дней')
print(f'  Следующий шаг:         Kaplan–Meier + Cox regression (lifelines)')